## 1. 뉴스제목 가져오기
* user-agent 요청헤더를 반드시 설정해야 한다.

In [4]:
# requests 라이브러리 설치여부 확인
! pip show requests

Name: requests
Version: 2.32.3
Summary: Python HTTP for Humans.
Home-page: https://requests.readthedocs.io
Author: Kenneth Reitz
Author-email: me@kennethreitz.org
License: Apache-2.0
Location: C:\Users\jeffv\anaconda3\Lib\site-packages
Requires: certifi, charset-normalizer, idna, urllib3
Required-by: aext-assistant-server, anaconda-auth, anaconda-catalogs, anaconda-client, anaconda-project, conda, conda-build, conda-repo-cli, conda_package_streaming, cookiecutter, datashader, jupyterlab_server, panel, requests-file, requests-toolbelt, Sphinx, streamlit, tldextract


In [5]:
# beautifulsoup4 라이브러리 설치여부 확인
! pip show beautifulsoup4

Name: beautifulsoup4
Version: 4.12.3
Summary: Screen-scraping library
Home-page: https://www.crummy.com/software/BeautifulSoup/bs4/
Author: 
Author-email: Leonard Richardson <leonardr@segfault.org>
License: MIT License
Location: C:\Users\jeffv\anaconda3\Lib\site-packages
Requires: soupsieve
Required-by: conda-build, nbconvert


In [6]:
# requests, bs4 import
import requests
import bs4
# BeautifulSoup 클래스 import
from bs4 import BeautifulSoup

In [7]:
# requests, bs4 버전 확인하기
print(f'requests 버전 = {requests.__version__}')
print(f'bs4 버전 = {bs4.__version__}')

requests 버전 = 2.32.3
bs4 버전 = 4.12.3


### 1. 뉴스 제목 추출하기

In [8]:
# IT/과학 뉴스 
req_param = {
    'sid': 105
}
# 
url = 'https://news.naver.com/section/{sid}'.format(**req_param)
print(url)

# 요청 헤더 설정 : 브라우저 정보
req_header = {
    'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36'
}

# requests 의 get() 함수 호출하기 
res = requests.get(url,headers=req_header)
print(res.status_code, res.ok)
# 응답(response)이 OK 이면
if res.ok: 
    # 응답 (response)에서 text 추출
    html = res.text
    # BeautifulSoup 객체 생성
    soup = BeautifulSoup(html,'html.parser')
    #print(soup)      
# CSS 선택자
# print(soup.select("div.sa_text a[href*='mnews/article']"))
    print(len(soup.select("div sa_text a[href*='https://n.news.naver.com/mnews']")))
    a_tags = soup.select("div sa_text a[href*='https://n.news.naver.com/mnews']") #속성찾기?
    print(type(a_tags))
# <a> 태그 리스트 순회하기    
    for a_tag in a_tags:
         title = a_tag.tex.strip() #<a>제목</a>
         link = a_tag['href']  #<a href="">title</a>
         print(title, link)
# 응답(response)이 Error 이면 status code 출력    
else:
     print(f'Error Code = {res.status_code}')

https://news.naver.com/section/105
200 True
0
<class 'bs4.element.ResultSet'>


### 1.1 뉴스제목 추출하는 함수 선언하기

In [9]:
import requests
from bs4 import BeautifulSoup

section_dict = {'정치': 100, '경제':101, '사회':102, '생활/문화':103, '세계':104, 'IT/과학':105 }

def print_news(section_name):  #print_new(103,'생활/문화') 
    sid = section_dict.get(section_name,'사회')
    url = f'https://news.naver.com/section/{sid}'
    print(f'{section_name} 뉴스 {url}')

    req_header = {
    'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36'
    }

    res = requests.get(url,headers=req_header)
    print(res.status_code, res.ok)
    if res.ok: 
        html = res.text
        soup = BeautifulSoup(html,'html.parser')
        print(len(soup.select("div sa_text a[href*='https://n.news.naver.com/mnews']")))
        a_tags = soup.select("div sa_text a[href*='https://n.news.naver.com/mnews']")
        print(type(a_tags))
        for a_tag in a_tags:
            title = a_tag.tex.strip()
            link = a_tag['href']  
            print(title, link)
    else:
        print(f'Error Code = {res.status_code}')

In [10]:
print_news('경제')

경제 뉴스 https://news.naver.com/section/101
200 True
0
<class 'bs4.element.ResultSet'>


### 2. Image 다운로드
* referer 요청 헤더를 반드시 설정해야 한다.

In [11]:
import requests
import os

req_header = {
    'referer':'https://comic.naver.com/webtoon/detail?titleId=812354&no=208&week=sun',
    'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/144.0.0.0 Safari/537.3'
}

img_urls = [
    'https://image-comic.pstatic.net/webtoon/812354/208/20251223155029_0d14a9e6cdd6b23342f96d1bd33d01c3_IMAG01_1.jpg',
    'https://image-comic.pstatic.net/webtoon/812354/208/20251223155029_0d14a9e6cdd6b23342f96d1bd33d01c3_IMAG01_2.jpg',
    'https://image-comic.pstatic.net/webtoon/812354/208/20251223155029_0d14a9e6cdd6b23342f96d1bd33d01c3_IMAG01_3.jpg'
]

for img_url in img_urls:
    # requests 의 get(url, headers) 함수 호출하기 
    res = requests.get(img_url, headers=req_header)
    #print(res.status_code)
    # binary 응답 데이터 가져오기
    img_data = res.content
    #print(type(img_data))
    # url에서 파일명만 추출하기
    file_name = os.path.basename(img_url)
    #print(file_name)    
    # binary data를 file에 write하기
    with open(file_name,'wb') as file:
        print(f'Writing to {file_name}({len(img_data):,} bytes)')
        file.write(img_data)


Writing to 20251223155029_0d14a9e6cdd6b23342f96d1bd33d01c3_IMAG01_1.jpg(124,462 bytes)
Writing to 20251223155029_0d14a9e6cdd6b23342f96d1bd33d01c3_IMAG01_2.jpg(141,956 bytes)
Writing to 20251223155029_0d14a9e6cdd6b23342f96d1bd33d01c3_IMAG01_3.jpg(170,913 bytes)


* 현재 요청된 페이지의 image 모두 다운로드 해보기

In [1]:
import requests
from bs4 import BeautifulSoup
import os

webtoon_url = 'https://comic.naver.com/webtoon/detail?titleId=812354&no=208&week=sun'

req_header = {
    'referer': webtoon_url,
    'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/144.0.0.0 Safari/537.3'
}

res = requests.get(webtoon_url, headers = req_header)
if res.ok:
    soup = BeautifulSoup(res.text,'html.parser')
    print(len(soup.select("img[src$='.jpg']")))
    print(len(soup.select("img[src*='IMAG01']")))
    #ResultSet 객체
    img_tags = soup.select("img[src*='IMAG01']")

    # 기존방식
    img_url_list = []   #['',''] 
    for img_tag in img_tags:
        img_url_list.append(img_tag['src'])
    print(img_url_list[:2])
    
    # Pythonic 방식 - List Comprehension 
    img_url_list2 = [img_tag['src'] for img_tag in img_tags]
    print(img_url_list2[:2])

    imgdir_name = 'img'
    if not os.path.isdir(imgdir_name):
        os.mkdir(imgdir_name)

    for img_url in img_url_list2:
        # requests 의 get(url, headers) 함수 호출하기 
        res = requests.get(img_url, headers=req_header)
        # binary 응답 데이터 가져오기
        img_data = res.content    

        #img/xxxIMG01.jpg
        file_path = os.path.join(imgdir_name,os.path.basename(img_url))
        # binary data를 file에 write하기
        with open(file_path,'wb') as file:
            print(f'Writing to {file_path}({len(img_data):,} bytes)')
            file.write(img_data)


263
13
['https://image-comic.pstatic.net/webtoon/812354/208/20251223155029_0d14a9e6cdd6b23342f96d1bd33d01c3_IMAG01_1.jpg', 'https://image-comic.pstatic.net/webtoon/812354/208/20251223155029_0d14a9e6cdd6b23342f96d1bd33d01c3_IMAG01_2.jpg']
['https://image-comic.pstatic.net/webtoon/812354/208/20251223155029_0d14a9e6cdd6b23342f96d1bd33d01c3_IMAG01_1.jpg', 'https://image-comic.pstatic.net/webtoon/812354/208/20251223155029_0d14a9e6cdd6b23342f96d1bd33d01c3_IMAG01_2.jpg']
Writing to img\20251223155029_0d14a9e6cdd6b23342f96d1bd33d01c3_IMAG01_1.jpg(124,462 bytes)
Writing to img\20251223155029_0d14a9e6cdd6b23342f96d1bd33d01c3_IMAG01_2.jpg(141,956 bytes)
Writing to img\20251223155029_0d14a9e6cdd6b23342f96d1bd33d01c3_IMAG01_3.jpg(170,913 bytes)
Writing to img\20251223155029_0d14a9e6cdd6b23342f96d1bd33d01c3_IMAG01_4.jpg(164,871 bytes)
Writing to img\20251223155029_0d14a9e6cdd6b23342f96d1bd33d01c3_IMAG01_5.jpg(132,142 bytes)
Writing to img\20251223155029_0d14a9e6cdd6b23342f96d1bd33d01c3_IMAG01_6.jpg(

#### 리팩토링 코드
*-* 현재 요청된 페이지의 image 모두 다운로드 해보기 

In [ ]:
import requests
from bs4 import BeautifulSoup
import os

# 기본 설정
url = 'https://comic.naver.com/webtoon/detail?titleId=833255&no=3&week=tue'
req_header = {
    'referer': url, 
    'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/144.0.0.0 Safari/537.3'
}
imgdir_name = 'img'

# 이미지 저장 폴더가 없으면 생성 ( sub 디렉토리도 생성 )
os.makedirs(imgdir_name, exist_ok=True)

# 웹 페이지 요청 및 확인
res = requests.get(url)
if not res.ok:
    print(f'Error Code = {res.status_code}')
    exit()

# 이미지 URL 추출
soup = BeautifulSoup(res.text, 'html.parser')
img_url_list = [img_tag['src'] for img_tag in soup.select("img[src*='IMAG01']")]

# 이미지 다운로드
for img_url in img_url_list:
    res = requests.get(img_url, headers=req_header)
    if res.ok:
        img_data = res.content
        file_path = os.path.join(imgdir_name, os.path.basename(img_url))
        with open(file_path, 'wb') as file:
            print(f'Writing to {file_path} ({len(img_data):,} bytes)')
            file.write(img_data)
    else:
        print(f'Error Code = {res.status_code} for {img_url}')

### 3. 파일 업로드 하기
* http://httpbin.org/post 업로드 요청을 할 수 있는 url

In [3]:
import requests

upload_files = {
    'img1': open('img/f1.jpg','rb'),
    'img2': open('img/f2.jpg','rb'),
}
print(upload_files)

url = 'http://httpbin.org/post'
# file 업로드 하려면 requests의 post 함수에 files 속성을 사용합니다.
res = requests.post(url, files=upload_files)
print(res.status_code)
print(res.json()['files']['img1'])

{'img1': <_io.BufferedReader name='img/f1.jpg'>, 'img2': <_io.BufferedReader name='img/f2.jpg'>}
200
...


### .env 파일에서 환경변수 읽어오기
* CLIENT_ID와 CLIENT_SECRET 환경변수 읽어오기

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

CLIENT_ID = os.getenv("CLIENT_ID")
print(CLIENT_ID[:4])

CLIENT_SECRET = os.getenv("CLIENT_SECRET")
print(CLIENT_SECRET[:4])

ok3n
IFlQ


### 4. 캡챠(이미지) API 호출하기
* urllib 사용
* [네이버개발자가이드](https://developers.naver.com/docs/utils/captcha/examples/)
* 1. 캡차 키 발급 요청
  2. 캡차 이미지 요청
  3. 사용자 입력값 검증 요청

In [12]:
# 캡차 키 발급 요청
import os
import sys
import urllib.request


code = "0"
url = f"https://openapi.naver.com/v1/captcha/nkey?code={code}"
request = urllib.request.Request(url)
request.add_header("X-Naver-Client-Id",CLIENT_ID)
request.add_header("X-Naver-Client-Secret",CLIENT_SECRET)
response = urllib.request.urlopen(request)
rescode = response.getcode()
if(rescode==200):
    response_body = response.read()
    print(response_body.decode('utf-8'))
else:
    print("Error Code:" + rescode)

{"key":"oUdWSzUPWlp0aEtB"}


In [13]:
# 캡차 이미지 요청
import os
import sys
import urllib.request

key = "oUdWSzUPWlp0aEtB" # 캡차 Key 값
url = f"https://openapi.naver.com/v1/captcha/ncaptcha.bin?key={key}"
request = urllib.request.Request(url)
request.add_header("X-Naver-Client-Id",CLIENT_ID)
request.add_header("X-Naver-Client-Secret",CLIENT_SECRET)
response = urllib.request.urlopen(request)
rescode = response.getcode()
if(rescode==200):
    print("캡차 이미지 저장")
    response_body = response.read()
    with open('captcha.jpg', 'wb') as f:
        f.write(response_body)
else:
    print("Error Code:" + rescode)

캡차 이미지 저장


In [14]:
#  사용자 입력값 검증 요청
import urllib.request


code = "1"
key = "oUdWSzUPWlp0aEtB"
value = "6VC24F5B"

url = f"https://openapi.naver.com/v1/captcha/nkey?code={code}&key={key}&value={value}"
request = urllib.request.Request(url)
request.add_header("X-Naver-Client-Id",CLIENT_ID)
request.add_header("X-Naver-Client-Secret",CLIENT_SECRET)
response = urllib.request.urlopen(request)
rescode = response.getcode()
if(rescode==200):
    response_body = response.read()
    print(response_body.decode('utf-8'))
else:
    print("Error Code:" + rescode)


{"result":true,"responseTime":65.33}


### 5-1. 블로그 검색하기
* 네이버 개발자센터에서 제공하는 검색 API 사용하기

In [ ]:
import pprint
pprint.pprint("")

from pprint import pprint
pprint()

In [16]:
import requests
from pprint import pprint

headers = {
    'X-Naver-Client-Id': CLIENT_ID,
    'X-Naver-Client-Secret': CLIENT_SECRET,
}

payload = {
    'query': '파이썬',
    'display': 100,
    'sort': 'sim'
}

#key1=value1&key2=value2&key3=value3
#https://openapi.naver.com/v1/search/blog.json?query=query+&display=100&
url = 'https://openapi.naver.com/v1/search/blog.json'

# requests get(url, params, headers) 요청 
res = requests.get(url, params=payload, headers=headers)

# json() 함수로 응답 결과 가져오기 [{},{},{}]
# 'title' , 'bloggername' , 'description' , 'bloggerlink' , 'link'
print(len(res.json()['items'])) 
pprint(res.json()['items'])

#items_data = res.json()['items']
# 'title' , 'bloggername' , 'description' , 'bloggerlink' , 'link'

100
[{'bloggerlink': 'https://blog.naver.com/1218seok',
  'bloggername': '소소한 월급러, 티끌도 소중해',
  'description': '들어보았떤 <b>파이썬</b> 독학을 검색하기 시작했어요. 그런데 막상... 아이티동스쿨의 '
                 '<b>파이썬</b>+R 패키지를 수강하게 됐습니다. 이 업체는... 아이티동스쿨 AI <b>파이썬</b>+R '
                 '패키지, 어떤 강의인가요? 항목 내용 패키지명 AI... ',
  'link': 'https://blog.naver.com/1218seok/224345671087',
  'postdate': '20260714',
  'title': '아이티동스쿨 AI <b>파이썬</b> 인강 후기, 비전공 직장인의 <b>파이썬</b>+R 독학 ....'},
 {'bloggerlink': 'https://blog.naver.com/dean4001',
  'bloggername': "고양이집사 리샤's 포스트 IT",
  'description': '데이터분석 입문자를 위한 <b>파이썬</b> vs 엑셀 선택 기준 데이터분석을 처음 시작할 때 '
                 '<b>파이썬</b>부터 배워야... 때 <b>파이썬</b>으로 확장하는 순서가 좋습니다. '
                 '<b>파이썬</b>은 강력한 도구지만, 데이터가 행과 열로 정리되는... ',
  'link': 'https://blog.naver.com/dean4001/224337018525',
  'postdate': '20260705',
  'title': '데이터분석 처음이면 <b>파이썬</b>보다 엑셀부터?'},
 {'bloggerlink': 'https://blog.naver.com/yerin0n_off',
  'bloggername': '칼퇴하는 예린',
  'description': '<b>파이썬</b>(PYTHON) 직

### 5-2. 블로그 검색하기
* 네이버 API HUB (NAVER Cloud)에서 제공하는 검색 API 사용하기

In [17]:
from dotenv import load_dotenv
import os

load_dotenv()

CLIENT_ID = os.getenv("CLIENT_ID")
print(CLIENT_ID[:4])

CLIENT_SECRET = os.getenv("CLIENT_SECRET")
print(CLIENT_SECRET[:4])

ok3n
IFlQ


In [ ]:
import requests
from pprint import pprint

headers = {
    'X-NCP-APIGW-API-KEY-ID': CLIENT_ID,
    'X-NCP-APIGW-API-KEY': CLIENT_SECRET,
}

payload = {
    'query': '파이썬',
    'display': 100,
    'sort': 'sim'
}
domain_url = 'https://naverapihub.apigw.ntruss.com'
url = f'{domain_url}/search/v1/blog'

# requests get(url, params, headers) 요청 
res = requests.get(url, params=payload, headers=headers)
# json() 함수로 응답 결과 가져오기 [{},{},{}]
print(len(res.json()['items'])) 
pprint(res.json()['items'])

[]
items_data = res.json()['items']
# 'title' , 'bloggername' , 'description' , 'bloggerlink' , 'link'

In [ ]:
# data/blog.json 파일 생성하기
import json

with open('data/blog.json','w', encoding='utf-8') as file:
    json.dump(items_data, file)

In [ ]:
import pandas as pd

print(pd.__version__)

blog_data = pd.read_json('data/blog.json')
print(blog_data.shape)

In [ ]:
blog_data.head()

In [ ]:
blog_data.columns

In [ ]:

blog_data['bloggername'].unique()